# GCP Compute Engine 인스턴스 생성 및 설정 예제

이 주피터 노트북은 Google Cloud SDK(`gcloud`)를 사용하여 다음 작업을 단계별로 수행합니다:

1. **동일 사양 글로벌 리전별 요금 비교 및 최저가 Top 3 분석**
2. **Compute Engine VM 인스턴스 생성** (`instance-20260914-054821` 또는 최저가 리전)
3. **Google Cloud Ops Agent 설정 파일(`config.yaml`) 생성**
4. **Ops Agent 정책 생성 및 적용**
5. **스냅샷 스케줄 정책(`default-schedule-1`) 생성**
6. **인스턴스 부팅 디스크에 스냅샷 스케줄 정책 연결**
7. **(부록) 원본 통합 셸 스크립트 실행 셀**
8. **(부록) 리소스 확인 및 과금 방지(삭제) 셀**

---
### 사전 준비 사항
- Google Cloud SDK (`gcloud`) 설치 및 인증 완료 (`gcloud auth login`)
- 기본 프로젝트 설정 (`gcloud config set project iceu-songpa11`)

## 0. 공통 헬퍼 함수 및 환경 확인
Windows(PowerShell/cmd) 및 Linux/macOS 환경 모두에서 `gcloud` 명령어가 원활히 실행되도록 파이썬 `subprocess` 헬퍼 함수를 정의합니다.

In [20]:
import subprocess
import sys
import os

# ============================================================
# [공통 설정 변수]
# 실습 환경에 맞춰 프로젝트 ID, 영역, 인스턴스명을 설정합니다.
# ============================================================
PROJECT_ID = "iceu-songpa11"
ZONE = "us-central1-a"
REGION = "us-central1"
INSTANCE_NAME = "instance-20260914-054821"
SNAPSHOT_SCHEDULE_NAME = "default-schedule-1"
OPS_POLICY_NAME = f"goog-ops-agent-v2-template-1-7-0-{ZONE}"
SERVICE_ACCOUNT = "976675812314-compute@developer.gserviceaccount.com"

def run_cmd(command_str):
    print(f"[실행 명령어]\n{command_str}\n")
    result = subprocess.run(command_str, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if result.stdout:
        print("[표준 출력]:")
        print(result.stdout)
    if result.stderr:
        print("[상태 / 메시지]:")
        print(result.stderr)
    if result.returncode == 0:
        print("[결과] 성공적으로 완료되었습니다.\n")
    else:
        print(f"[결과] 실패 (종료 코드: {result.returncode})\n")
    return result

# 현재 활성화된 gcloud 설정 확인
run_cmd("gcloud config list")

[실행 명령어]
gcloud config list

[표준 출력]:
[accessibility]
screen_reader = False
[compute]
region = us-central1
zone = us-central1-c
[core]
account = songpa11@iceu.kr
disable_usage_reporting = True
project = iceu-songpa11

[상태 / 메시지]:

Your active configuration is: [default]
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.

[결과] 성공적으로 완료되었습니다.



CompletedProcess(args='gcloud config list', returncode=0, stdout='[accessibility]\nscreen_reader = False\n[compute]\nregion = us-central1\nzone = us-central1-c\n[core]\naccount = songpa11@iceu.kr\ndisable_usage_reporting = True\nproject = iceu-songpa11\n', stderr='\nYour active configuration is: [default]\n[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.\n')

## [비용 분석] 동일 사양 Compute Engine 최저가 Region Top 3 분석

현재 구성된 인스턴스와 동일한 옵션 조건에서의 리전별 비용을 비교합니다:
- **머신 유형**: `e2-medium` (2 vCPU, 4GB RAM)
- **부팅 디스크**: 10GB `pd-balanced` (밸런스드 영구 디스크)
- **프로비저닝 모델**: `STANDARD` (상시 가동 기준, 월 730시간)
- **네트워크 티어**: `PREMIUM`
- **OS**: Debian 13 (무료 오픈소스 라이선스)

### 🏆 가장 저렴한 Region Top 3 (동일 최저가 그룹)
Google Cloud는 Tier 1 핵심 미국 리전들에 동일한 전 세계 최저 요금을 적용하고 있습니다.

1. **1위 (공동): `us-central1` (Iowa, 미국 중부) — [현재 노트북 기본 설정 리전]**
   - **VM 요금 (`e2-medium`)**: **$0.03350 / 시간** (약 **$24.46 / 월**)
   - **디스크 요금 (10GB `pd-balanced`)**: **$1.00 / 월** ($0.10 / GB·월)
   - **월 총 예상 비용**: **$25.46 / 월** (시간당 약 **$0.0349**)

2. **2위 (공동): `us-east1` (South Carolina, 미국 동부)**
   - **VM 요금 (`e2-medium`)**: **$0.03350 / 시간** (약 **$24.46 / 월**)
   - **디스크 요금 (10GB `pd-balanced`)**: **$1.00 / 월** ($0.10 / GB·월)
   - **월 총 예상 비용**: **$25.46 / 월** (시간당 약 **$0.0349**)

3. **3위 (공동): `us-west1` (Oregon, 미국 서부)**
   - **VM 요금 (`e2-medium`)**: **$0.03350 / 시간** (약 **$24.46 / 월**)
   - **디스크 요금 (10GB `pd-balanced`)**: **$1.00 / 월** ($0.10 / GB·월)
   - **월 총 예상 비용**: **$25.46 / 월** (시간당 약 **$0.0349**)

> *(참고: `us-east4`(버지니아 북부), `us-east5`(콜럼버스) 또한 동일하게 $25.46/월로 최저가 그룹에 해당합니다.)*
> *(비교: 서울 리전 `asia-northeast3`은 월 약 $30.55로 미국 최저가 대비 약 20% 비쌉니다.)*

In [21]:
# [코드 1] 리전별 Compute Engine (e2-medium + 10GB pd-balanced) 비용 비교 및 Top 3 조회
import sys
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# GCP 공식 요금표 기준 리전별 데이터 (e2-medium 2 vCPU + 4GB RAM / 730시간 기준, pd-balanced 10GB)
regions_pricing = [
    {'rank': 1, 'region': 'us-central1', 'location': 'Iowa (미국 중부)', 'vm_hourly': 0.03350, 'disk_monthly': 1.00},
    {'rank': 2, 'region': 'us-east1', 'location': 'South Carolina (미국 동부)', 'vm_hourly': 0.03350, 'disk_monthly': 1.00},
    {'rank': 3, 'region': 'us-west1', 'location': 'Oregon (미국 서부)', 'vm_hourly': 0.03350, 'disk_monthly': 1.00},
    {'rank': 4, 'region': 'us-east4', 'location': 'N. Virginia (미국 동부)', 'vm_hourly': 0.03350, 'disk_monthly': 1.00},
    {'rank': 5, 'region': 'us-east5', 'location': 'Columbus (미국 동부)', 'vm_hourly': 0.03350, 'disk_monthly': 1.00},
    {'rank': 6, 'region': 'europe-west1', 'location': 'Belgium (유럽 벨기에)', 'vm_hourly': 0.03685, 'disk_monthly': 1.10},
    {'rank': 7, 'region': 'europe-north1', 'location': 'Finland (유럽 핀란드)', 'vm_hourly': 0.03685, 'disk_monthly': 1.10},
    {'rank': 8, 'region': 'asia-east1', 'location': 'Taiwan (아시아 대만)', 'vm_hourly': 0.03841, 'disk_monthly': 1.10},
    {'rank': 9, 'region': 'asia-northeast3', 'location': 'Seoul (한국 서울)', 'vm_hourly': 0.04020, 'disk_monthly': 1.20},
    {'rank': 10, 'region': 'southamerica-east1', 'location': 'Sao Paulo (남미 상파울루)', 'vm_hourly': 0.05140, 'disk_monthly': 1.50},
]

for item in regions_pricing:
    item['vm_monthly'] = round(item['vm_hourly'] * 730, 2)
    item['total_monthly'] = round(item['vm_monthly'] + item['disk_monthly'], 2)

print('=' * 85)
print('[GCP 리전별 요금 순위] 동일 옵션(e2-medium + 10GB pd-balanced) 기준')
print('=' * 85)
print(f"{'순위':<4} | {'리전(Region)':<18} | {'위치(Location)':<25} | {'시간당(VM)':<10} | {'월비용(Total)'}")
print('-' * 85)

for item in regions_pricing:
    is_top3 = '[TOP 3]' if item['rank'] <= 3 else ''
    is_current = '(현재 설정)' if item['region'] == 'us-central1' else ''
    print(f"{item['rank']:<4} | {item['region']:<18} | {item['location']:<25} | ${item['vm_hourly']:.5f}/h | ${item['total_monthly']:.2f}/월 {is_top3} {is_current}")

print('=' * 85)
top3 = regions_pricing[:3]
print('\n[결론] 가장 저렴한 최저가 Region Top 3:')
for i, r in enumerate(top3, 1):
    print(f"   {i}위: {r['region']} ({r['location']}) - 월 총 ${r['total_monthly']:.2f} (VM: ${r['vm_monthly']:.2f} + 디스크: ${r['disk_monthly']:.2f})")
print("\n[안내] 현재 인스턴스가 사용하는 'us-central1' 리전이 이미 글로벌 최저가 리전입니다!")


[GCP 리전별 요금 순위] 동일 옵션(e2-medium + 10GB pd-balanced) 기준
순위   | 리전(Region)         | 위치(Location)              | 시간당(VM)    | 월비용(Total)
-------------------------------------------------------------------------------------
1    | us-central1        | Iowa (미국 중부)              | $0.03350/h | $25.46/월 [TOP 3] (현재 설정)
2    | us-east1           | South Carolina (미국 동부)    | $0.03350/h | $25.46/월 [TOP 3] 
3    | us-west1           | Oregon (미국 서부)            | $0.03350/h | $25.46/월 [TOP 3] 
4    | us-east4           | N. Virginia (미국 동부)       | $0.03350/h | $25.46/월  
5    | us-east5           | Columbus (미국 동부)          | $0.03350/h | $25.46/월  
6    | europe-west1       | Belgium (유럽 벨기에)          | $0.03685/h | $28.00/월  
7    | europe-north1      | Finland (유럽 핀란드)          | $0.03685/h | $28.00/월  
8    | asia-east1         | Taiwan (아시아 대만)           | $0.03841/h | $29.14/월  
9    | asia-northeast3    | Seoul (한국 서울)             | $0.04020/h | $30.55/월  
10   | southamerica-east1 | Sao

In [22]:
import sys
# [코드 2] 최저가 리전을 선택하여 동일한 옵션으로 Compute Engine 생성하기

# 최저가 Top 3 리전 중 원하는 영역(Zone)을 선택하세요:
# 1) us-central1-a (Iowa, 현재 기본값)
# 2) us-east1-b (South Carolina)
# 3) us-west1-b (Oregon)

TARGET_ZONE = "us-central1-a"  # 최저가 1위 리전
INSTANCE_NAME = "instance-cheapest-region"
PROJECT_ID = "iceu-songpa11"

cmd_create_cheapest = f"""gcloud compute instances create {INSTANCE_NAME} \
    --project={PROJECT_ID} \
    --zone={TARGET_ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=976675812314-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any"""

if sys.platform == "win32":
    cmd_create_cheapest = cmd_create_cheapest.replace("\\\n", "^\n")

print(f"[안내] 최저가 영역({TARGET_ZONE})에 동일한 사양의 인스턴스 생성을 준비합니다.")
# 생성을 원할 때 아래 줄의 주석을 해제하고 실행하세요:
# run_cmd(cmd_create_cheapest)
print("인스턴스 생성 명령어 준비 완료.")

[안내] 최저가 영역(us-central1-a)에 동일한 사양의 인스턴스 생성을 준비합니다.
인스턴스 생성 명령어 준비 완료.


## 1. Compute Engine 인스턴스 생성
- **인스턴스 명**: `instance-20260914-054821`
- **프로젝트**: `iceu-songpa11`
- **영역**: `us-central1-a`
- **머신 유형**: `e2-medium` (2 vCPU, 4GB 메모리)
- **운영체제 이미지**: Debian 13 (`projects/debian-cloud/global/images/debian-13-trixie-v20260908`)
- **부팅 디스크**: 10GB `pd-balanced` (인스턴스 삭제 시 자동 삭제 설정)
- **라벨**: `goog-ops-agent-policy=v2-template-1-7-0`, `goog-ec-src=vm_add-gcloud`

In [23]:
cmd_create_instance = f"""gcloud compute instances create {INSTANCE_NAME} \
    --project={PROJECT_ID} \
    --zone={ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account={SERVICE_ACCOUNT} \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any"""

# Windows 환경(cmd.exe)에서는 캐럿(^)으로 변환하여 안전하게 실행
if sys.platform == "win32":
    cmd_create_instance = cmd_create_instance.replace("\\\n", "^\n")

run_cmd(cmd_create_instance)

[실행 명령어]
gcloud compute instances create instance-20260914-054821     --project=iceu-songpa11     --zone=us-central1-a     --machine-type=e2-medium     --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default     --metadata=enable-osconfig=TRUE     --maintenance-policy=MIGRATE     --provisioning-model=STANDARD     --service-account=976675812314-compute@developer.gserviceaccount.com     --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append     --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054821,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced     --no-shielded-secure-boot     --shielded-vtpm     --shielded-integrity-monitoring     --labels=g

CompletedProcess(args='gcloud compute instances create instance-20260914-054821     --project=iceu-songpa11     --zone=us-central1-a     --machine-type=e2-medium     --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default     --metadata=enable-osconfig=TRUE     --maintenance-policy=MIGRATE     --provisioning-model=STANDARD     --service-account=976675812314-compute@developer.gserviceaccount.com     --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append     --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054821,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced     --no-shielded-secure-boot     --shielded-vtpm     --shielded-integrity-monitoring 

## 2. Ops Agent 정책 설정 파일 (`config.yaml`) 생성
Cloud Monitoring 및 Cloud Logging을 위한 Ops Agent 자동 설치 및 최신 버전 유지를 정의하는 `config.yaml` 파일을 작성합니다.

In [24]:
config_yaml_content = """agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
"""

with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml_content)

print("[생성 완료] config.yaml 내용:")
with open("config.yaml", "r", encoding="utf-8") as f:
    print(f.read())

[생성 완료] config.yaml 내용:
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0



## 3. Ops Agent 정책 생성 및 적용
작성한 `config.yaml` 설정 파일을 기반으로 `us-central1-a` 영역에 Ops Agent 정책을 등록합니다.

In [25]:
cmd_ops_policy = f"""gcloud compute instances ops-agents policies create {OPS_POLICY_NAME} \
    --project={PROJECT_ID} \
    --zone={ZONE} \
    --file=config.yaml"""

if sys.platform == "win32":
    cmd_ops_policy = cmd_ops_policy.replace("\\\n", "^\n")

res = run_cmd(cmd_ops_policy)
if res.returncode != 0 and ("ALREADY_EXISTS" in res.stderr or "already exists" in res.stderr):
    print("[안내] Ops Agent 정책이 이미 등록되어 있습니다. 기존 정책 정보를 확인합니다:")
    run_cmd(f"gcloud compute instances ops-agents policies describe {OPS_POLICY_NAME} --project={PROJECT_ID} --zone={ZONE}")


[실행 명령어]
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a     --project=iceu-songpa11     --zone=us-central1-a     --file=config.yaml

[상태 / 메시지]:
ERROR: (gcloud.compute.instances.ops-agents.policies.create) ALREADY_EXISTS: Requested entity already exists

[결과] 실패 (종료 코드: 1)



CompletedProcess(args='gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a     --project=iceu-songpa11     --zone=us-central1-a     --file=config.yaml', returncode=1, stdout='', stderr='ERROR: (gcloud.compute.instances.ops-agents.policies.create) ALREADY_EXISTS: Requested entity already exists\n')

## 4. 디스크 스냅샷 스케줄 정책 생성
- **정책 이름**: `default-schedule-1`
- **리전**: `us-central1`
- **보관 주기**: 최대 14일 (`--max-retention-days=14`)
- **스케줄**: 매일 23:00 UTC 기준 1일 1회 실행
- **소스 디스크 삭제 시 동작**: 자동 생성된 스냅샷 유지 (`keep-auto-snapshots`)

In [26]:
cmd_snapshot_policy = f"""gcloud compute resource-policies create snapshot-schedule {SNAPSHOT_SCHEDULE_NAME} \
    --project={PROJECT_ID} \
    --region={REGION} \
    --max-retention-days=14 \
    --on-source-disk-delete=keep-auto-snapshots \
    --daily-schedule \
    --start-time=23:00"""

if sys.platform == "win32":
    cmd_snapshot_policy = cmd_snapshot_policy.replace("\\\n", "^\n")

res = run_cmd(cmd_snapshot_policy)
if res.returncode != 0 and ("ALREADY_EXISTS" in res.stderr or "already exists" in res.stderr):
    print("[안내] 스냅샷 정책이 이미 등록되어 있습니다. 기존 정책 정보를 확인합니다:")
    run_cmd(f"gcloud compute resource-policies describe {SNAPSHOT_SCHEDULE_NAME} --project={PROJECT_ID} --region={REGION}")


[실행 명령어]
gcloud compute resource-policies create snapshot-schedule default-schedule-1     --project=iceu-songpa11     --region=us-central1     --max-retention-days=14     --on-source-disk-delete=keep-auto-snapshots     --daily-schedule     --start-time=23:00

[상태 / 메시지]:
Created [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1].

[결과] 성공적으로 완료되었습니다.



CompletedProcess(args='gcloud compute resource-policies create snapshot-schedule default-schedule-1     --project=iceu-songpa11     --region=us-central1     --max-retention-days=14     --on-source-disk-delete=keep-auto-snapshots     --daily-schedule     --start-time=23:00', returncode=0, stdout='', stderr='Created [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1].\n')

## 5. 인스턴스 부팅 디스크에 스냅샷 스케줄 정책 연결
방금 생성한 `default-schedule-1` 정책을 `instance-20260914-054821` 인스턴스의 디스크에 연결합니다.

In [27]:
cmd_add_policy = f"""gcloud compute disks add-resource-policies {INSTANCE_NAME} \
    --project={PROJECT_ID} \
    --zone={ZONE} \
    --resource-policies=projects/{PROJECT_ID}/regions/{REGION}/resourcePolicies/{SNAPSHOT_SCHEDULE_NAME}"""

if sys.platform == "win32":
    cmd_add_policy = cmd_add_policy.replace("\\\n", "^\n")

run_cmd(cmd_add_policy)

[실행 명령어]
gcloud compute disks add-resource-policies instance-20260914-054821     --project=iceu-songpa11     --zone=us-central1-a     --resource-policies=projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1

[상태 / 메시지]:
Updated [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/zones/us-central1-a/disks/instance-20260914-054821].

[결과] 성공적으로 완료되었습니다.



CompletedProcess(args='gcloud compute disks add-resource-policies instance-20260914-054821     --project=iceu-songpa11     --zone=us-central1-a     --resource-policies=projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1', returncode=0, stdout='', stderr='Updated [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/zones/us-central1-a/disks/instance-20260914-054821].\n')

## 6. [부록] 원본 통합 셸 명령어 단일 실행 (Bash / Colab / Linux 전용)
Linux, Colab, Mac 또는 Git Bash 환경에서는 아래 `%%bash` 셀을 통해 원본 한 줄 명령어를 그대로 일괄 실행할 수도 있습니다.

In [ ]:
%%bash

# ============================================================
# [부록 6] 원본 통합 셸 명령어 실행
# Bash / Colab / Linux / WSL 환경용
#
# ※ 이 셀은 원본 통합 스크립트를 일괄 실행하는 부록 셀입니다.
# ※ Windows Jupyter 환경에서는 %%bash 대신 앞선 파이썬 셀들을 순차 실행하세요.
# ※ WSL 환경에서 Windows gcloud 인증을 공유하려면 아래 환경변수 주석을 해제하세요:
#    export CLOUDSDK_CONFIG="/mnt/c/Users/USER/AppData/Roaming/gcloud"
# ============================================================

PROJECT_ID="iceu-songpa11"
ZONE="us-central1-a"
REGION="us-central1"
INSTANCE_NAME="instance-20260915-143200"
POLICY_NAME="goog-ops-agent-v2-template-1-7-0-us-central1-a"
SCHEDULE_NAME="default-schedule-1"
SERVICE_ACCOUNT="976675812314-compute@developer.gserviceaccount.com"

echo "============================================================"
echo "GCP 인증 상태"
echo "============================================================"
gcloud auth list

echo
echo "현재 계정:"
gcloud config get-value account

echo
echo "현재 프로젝트:"
gcloud config get-value project

echo
echo "============================================================"
echo "1. Compute Engine VM 생성"
echo "============================================================"
gcloud compute instances create ${INSTANCE_NAME} \
    --project=${PROJECT_ID} \
    --zone=${ZONE} \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=${SERVICE_ACCOUNT} \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=${INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any

echo
echo "============================================================"
echo "2. Ops Agent 정책 설정 파일 생성"
echo "============================================================"
cat << 'EOF' > config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
EOF
cat config.yaml

echo
echo "============================================================"
echo "3. Ops Agent 정책 생성"
echo "============================================================"
if gcloud compute instances ops-agents policies describe ${POLICY_NAME} --project=${PROJECT_ID} --zone=${ZONE} >/dev/null 2>&1; then
    echo "Ops Agent 정책이 이미 존재합니다. 생성 단계를 건너뜁니다."
else
    gcloud compute instances ops-agents policies create ${POLICY_NAME} \
        --project=${PROJECT_ID} \
        --zone=${ZONE} \
        --file=config.yaml
fi

echo
echo "============================================================"
echo "4. Snapshot Schedule 생성"
echo "============================================================"
if gcloud compute resource-policies describe ${SCHEDULE_NAME} --project=${PROJECT_ID} --region=${REGION} >/dev/null 2>&1; then
    echo "Snapshot Schedule이 이미 존재합니다. 생성 단계를 건너뜁니다."
else
    gcloud compute resource-policies create snapshot-schedule ${SCHEDULE_NAME} \
        --project=${PROJECT_ID} \
        --region=${REGION} \
        --max-retention-days=14 \
        --on-source-disk-delete=keep-auto-snapshots \
        --daily-schedule \
        --start-time=23:00
fi

echo
echo "============================================================"
echo "5. VM 디스크에 Snapshot Policy 연결"
echo "============================================================"
gcloud compute disks add-resource-policies ${INSTANCE_NAME} \
    --project=${PROJECT_ID} \
    --zone=${ZONE} \
    --resource-policies=projects/${PROJECT_ID}/regions/${REGION}/resourcePolicies/${SCHEDULE_NAME}

echo
echo "============================================================"
echo "6. 최종 확인"
echo "============================================================"
echo "[VM 상태]"
gcloud compute instances list --project=${PROJECT_ID} --filter="name=${INSTANCE_NAME}"

echo "[Snapshot Policy]"
gcloud compute resource-policies describe ${SCHEDULE_NAME} --project=${PROJECT_ID} --region=${REGION}

echo
echo "============================================================"
echo "모든 작업이 완료되었습니다."
echo "============================================================"


Updated property [core/account].
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


GCP 인증 상태
 Credentialed Accounts
ACTIVE  ACCOUNT
*       songpa11@iceu.kr



To set the active account, run:
    $ gcloud config set account `ACCOUNT`




현재 계정:
songpa11@iceu.kr

현재 프로젝트:
iceu-songpa11

1. Compute Engine VM 생성


ERROR: (gcloud.compute.instances.create) Could not fetch resource:
 - The resource 'projects/iceu-songpa11/zones/us-central1-a/instances/instance-20260914-054821' already exists




2. Ops Agent 정책 설정

3. Ops Agent 정책 생성
Ops Agent 정책이 이미 존재합니다. 생성 단계를 건너뜁니다.

4. Snapshot Schedule 생성
Snapshot Schedule이 이미 존재합니다. 생성 단계를 건너뜁니다.

5. VM 디스크에 Snapshot Policy 연결


Updated [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/zones/us-central1-a/disks/instance-20260914-054821].



6. 최종 확인

[VM 상태]
NAME                      STATUS   MACHINE_TYPE  ZONE
instance-20260914-054821  RUNNING  e2-medium     us-central1-a

[Snapshot Policy]
creationTimestamp: '2026-09-14T00:48:23.281-07:00'
id: '5880636660261722152'
kind: compute#resourcePolicy
name: default-schedule-1
region: https://www.googleapis.com/compute/v1/projects/iceu-songpa11/regions/us-central1
selfLink: https://www.googleapis.com/compute/v1/projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1
snapshotSchedulePolicy:
  retentionPolicy:
    maxRetentionDays: 14
    onSourceDiskDelete: KEEP_AUTO_SNAPSHOTS
  schedule:
    dailySchedule:
      daysInCycle: 1
      duration: PT14400S
      startTime: 23:00
status: READY

모든 작업이 완료되었습니다.


## 7. [부록] 생성된 리소스 확인 및 삭제(과금 방지 가이드)
실습이나 테스트가 끝난 후 지속적인 클라우드 요금 청구를 방지하기 위해 생성된 리소스를 확인하고 안전하게 정리할 수 있습니다.

In [30]:
# 현재 리소스 상태 확인 (과금 유발 요소 전수 점검)
print("=== [1] Compute Engine 인스턴스 목록 ===")
run_cmd(f"gcloud compute instances list --project={PROJECT_ID}")

print("=== [2] 영구 디스크 목록 ===")
run_cmd(f"gcloud compute disks list --project={PROJECT_ID}")

print("=== [3] 스냅샷 정책 목록 ===")
run_cmd(f"gcloud compute resource-policies list --project={PROJECT_ID}")

print("=== [4] 생성된 디스크 스냅샷 데이터 목록 (용량 과금) ===")
run_cmd(f"gcloud compute snapshots list --project={PROJECT_ID}")

print("=== [5] 고정 외부 IP 주소 목록 (미연결 시 시간당 과금) ===")
run_cmd(f"gcloud compute addresses list --project={PROJECT_ID}")


=== [1] Compute Engine 인스턴스 목록 ===
[실행 명령어]
gcloud compute instances list --project=iceu-songpa11

[상태 / 메시지]:
Listed 0 items.

[결과] 성공적으로 완료되었습니다.

=== [2] 영구 디스크 목록 ===
[실행 명령어]
gcloud compute disks list --project=iceu-songpa11

[상태 / 메시지]:
Listed 0 items.

[결과] 성공적으로 완료되었습니다.

=== [3] 스냅샷 정책 목록 ===
[실행 명령어]
gcloud compute resource-policies list --project=iceu-songpa11

[상태 / 메시지]:
Listed 0 items.

[결과] 성공적으로 완료되었습니다.



CompletedProcess(args='gcloud compute resource-policies list --project=iceu-songpa11', returncode=0, stdout='', stderr='Listed 0 items.\n')

In [ ]:
# [주의] 아래 코드는 생성한 인스턴스와 스케줄 정책을 완전히 삭제합니다.
# 'Run All(모든 셀 실행)' 시 실수로 인스턴스가 즉시 삭제되는 것을 방지하기 위해 기본적으로 주석(#) 처리되어 있습니다.
# 리소스 정리를 원할 때 아래 각 줄의 주석(#)을 해제하고 이 셀만 단독 실행하세요.

# 1) VM 인스턴스 삭제 (auto-delete=yes 옵션에 의해 부팅 디스크도 함께 자동 삭제됨)
# run_cmd(f"gcloud compute instances delete {INSTANCE_NAME} --zone={ZONE} --project={PROJECT_ID} --quiet")

# 2) 스냅샷 스케줄 정책 삭제
# run_cmd(f"gcloud compute resource-policies delete {SNAPSHOT_SCHEDULE_NAME} --region={REGION} --project={PROJECT_ID} --quiet")

# 3) Ops Agent 정책 삭제
# run_cmd(f"gcloud compute instances ops-agents policies delete {OPS_POLICY_NAME} --zone={ZONE} --project={PROJECT_ID} --quiet")

# 4) 로컬 config.yaml 임시 파일 삭제
# if os.path.exists('config.yaml'):
#     os.remove('config.yaml')
#     print('config.yaml 삭제 완료')


[실행 명령어]
gcloud compute instances delete instance-20260914-054821 --zone=us-central1-a --project=iceu-songpa11 --quiet

[상태 / 메시지]:
Deleted [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/zones/us-central1-a/instances/instance-20260914-054821].

[결과] 성공적으로 완료되었습니다.

[실행 명령어]
gcloud compute resource-policies delete default-schedule-1 --region=us-central1 --project=iceu-songpa11 --quiet

[상태 / 메시지]:
Deleted [https://www.googleapis.com/compute/v1/projects/iceu-songpa11/regions/us-central1/resourcePolicies/default-schedule-1].

[결과] 성공적으로 완료되었습니다.

[실행 명령어]
gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --project=iceu-songpa11 --quiet

[상태 / 메시지]:
ERROR: (gcloud.compute.instances.ops-agents.policies.delete) Ops Agents policy [goog-ops-agent-v2-template-1-7-0-us-central1-a] not found

[결과] 실패 (종료 코드: 1)

config.yaml 삭제 완료
